In [1]:
from pathlib import Path
import pandas as pd

# Мы в папке notebooks, поэтому ../data/raw ведет в папку data/raw
RAW = Path('../data/raw')

# Твои сезоны
SEASONS = ['1415', '1516', '1617', '1718', '1819', '1920', 
           '2021', '2122', '2223', '2324', '2425', '2526', '2627']

parts = []
for s in SEASONS:
    path = RAW / f'SP1_{s}.csv'
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='latin-1')
    
    print(f"✅ Прочитан: {s} (строк: {df.shape[0]})")
    parts.append(df)

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\SP1_1415.csv'

In [2]:
import os
from pathlib import Path

RAW = Path('../data/raw')
print("Настоящие имена файлов в папке raw:")
for file in os.listdir(RAW):
    print(file)

Настоящие имена файлов в папке raw:
.gitkeep.txt
SP1_1415.csv.csv
SP1_1516.csv.csv
SP1_1617.csv.csv
SP1_1718.csv.csv
SP1_1819.csv.csv
SP1_1920.csv.csv
SP1_2021.csv.csv
SP1_2122.csv.csv
SP1_2223.csv.csv
SP1_2324.csv.csv
SP1_2425.csv.csv
SP1_2526.csv.csv
SP1_2627.csv.csv


In [3]:
from pathlib import Path
import pandas as pd

RAW = Path('../data/raw')

# Твои сезоны
SEASONS = ['1415', '1516', '1617', '1718', '1819', '1920', 
           '2021', '2122', '2223', '2324', '2425', '2526', '2627']

parts = []
for s in SEASONS:
    # Ищем файлы с двойным расширением .csv.csv
    path = RAW / f'SP1_{s}.csv.csv'
    
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='latin-1')
    
    print(f"✅ Прочитан: {s} (строк: {df.shape[0]})")
    parts.append(df)

✅ Прочитан: 1415 (строк: 380)
✅ Прочитан: 1516 (строк: 380)
✅ Прочитан: 1617 (строк: 380)
✅ Прочитан: 1718 (строк: 380)
✅ Прочитан: 1819 (строк: 380)
✅ Прочитан: 1920 (строк: 380)
✅ Прочитан: 2021 (строк: 380)
✅ Прочитан: 2122 (строк: 380)
✅ Прочитан: 2223 (строк: 380)
✅ Прочитан: 2324 (строк: 380)
✅ Прочитан: 2425 (строк: 380)
✅ Прочитан: 2526 (строк: 380)
✅ Прочитан: 2627 (строк: 99)


In [4]:
# Оставляем только те колонки, которые нужны для проекта
COLS = {
    'Date': 'date',
    'HomeTeam': 'home',
    'AwayTeam': 'away',
    'FTHG': 'home_goals',
    'FTAG': 'away_goals',
    'FTR': 'result',
}

parts2 = []
for s, d in zip(SEASONS, parts):
    # Выбираем нужные колонки и переименовываем их
    d = d[list(COLS)].rename(columns=COLS)
    # Добавляем колонку с названием сезона
    d['season'] = s
    parts2.append(d)

# Склеиваем все 13 таблиц в одну большую
df = pd.concat(parts2, ignore_index=True)

# Показываем первые 5 строк, чтобы проверить
df.head()

,date,home,away,home_goals,away_goals,result,season
0,23/08/14,Almeria,Espanol,1,1,D,1415
1,23/08/14,Granada,La Coruna,2,1,H,1415
2,23/08/14,Malaga,Ath Bilbao,1,0,H,1415
3,23/08/14,Sevilla,Valencia,1,1,D,1415
4,24/08/14,Barcelona,Elche,3,0,H,1415


In [5]:
# 1. Убираем пустые строки (если есть)
df = df.dropna(subset=['date','home','away','home_goals','away_goals','result']).copy()

# 2. Превращаем текст в настоящие даты
# dayfirst=True нужен, потому что в Европе пишут 25/08/2023 (день/месяц/год)
df['date'] = pd.to_datetime(df['date'], dayfirst=True, format='mixed')

# 3. Голы делаем целыми числами (чтобы не было 2.0)
df['home_goals'] = df['home_goals'].astype(int)
df['away_goals'] = df['away_goals'].astype(int)

# 4. Удаляем дубликаты (если один и тот же матч попал дважды)
df = df.drop_duplicates(subset=['date','home','away'])

# 5. Сортируем всё по дате (от старых к новым)
df = df.sort_values('date').reset_index(drop=True)

print('Всего матчей в таблице:', len(df))
df.head()

Всего матчей в таблице: 4659


,date,home,away,home_goals,away_goals,result,season
0,2014-08-23,Almeria,Espanol,1,1,D,1415
1,2014-08-23,Granada,La Coruna,2,1,H,1415
2,2014-08-23,Malaga,Ath Bilbao,1,0,H,1415
3,2014-08-23,Sevilla,Valencia,1,1,D,1415
4,2014-08-24,Barcelona,Elche,3,0,H,1415


In [6]:
# Проверяем, что в каждом сезоне по 380 матчей
print(df.groupby('season').size())

# Проверяем, что победа (H) совпадает с тем, что у хозяев больше голов
import numpy as np
expected = np.where(df['home_goals'] > df['away_goals'], 'H', 
                    np.where(df['home_goals'] == df['away_goals'], 'D', 'A'))
bad = df[df['result'] != expected]
print('Ошибок в счете:', len(bad))

season
1415    380
1516    380
1617    380
1718    380
1819    380
1920    380
2021    380
2122    380
2223    380
2324    380
2425    380
2526    380
2627     99
dtype: int64
Ошибок в счете: 0


In [7]:
# 1. Убираем пустые строки (если есть)
df = df.dropna(subset=['date','home','away','home_goals','away_goals','result']).copy()

# 2. Превращаем текст в настоящие даты
# dayfirst=True нужен, потому что в Европе пишут 25/08/2023 (день/месяц/год)
df['date'] = pd.to_datetime(df['date'], dayfirst=True, format='mixed')

# 3. Голы делаем целыми числами (чтобы не было 2.0)
df['home_goals'] = df['home_goals'].astype(int)
df['away_goals'] = df['away_goals'].astype(int)

# 4. Удаляем дубликаты (если один и тот же матч попал дважды)
df = df.drop_duplicates(subset=['date','home','away'])

# 5. Сортируем всё по дате (от старых к новым)
df = df.sort_values('date').reset_index(drop=True)

print('Всего матчей в таблице:', len(df))
df.head()

Всего матчей в таблице: 4659


,date,home,away,home_goals,away_goals,result,season
0,2014-08-23,Almeria,Espanol,1,1,D,1415
1,2014-08-23,Granada,La Coruna,2,1,H,1415
2,2014-08-23,Malaga,Ath Bilbao,1,0,H,1415
3,2014-08-23,Sevilla,Valencia,1,1,D,1415
4,2014-08-24,Barcelona,Elche,3,0,H,1415


In [8]:
# Создаем папку processed, если ее еще нет
Path('../data/processed').mkdir(parents=True, exist_ok=True)

# Сохраняем нашу чистую таблицу в файл
df.to_csv('../data/processed/matches_clean.csv', index=False)
print('Файл matches_clean.csv успешно сохранен!')

Файл matches_clean.csv успешно сохранен!
